# Problem statement

A media or news company receive a large volume of articles everyday. To manually sort each paper into categories (such as business, sport, politics) is time-consuming, expensive and inconsistent at large scale.

The core business problem is: *How can the company automatically organize large number of incoming articles into correct category quickly and accurately?*

# Why this is important

Accurate article classification is important since it can support several business functions such as content organization, improving user experience, personalization and recommendation, and analytics and insights.

Once this problem is solved, the company have lower editorial workflow, faster content processing, better search and navigation, stronger recommendation systems, and better audience engagement. For instance, once articles get labeled automatically, a news website can quickly put them into right section and notify interested readers, which increase click-through rates.

# How relevant data would be collected

To build the current system, a dataset of [BBC full text documents](https://www.kaggle.com/datasets/alfathterry/bbc-full-text-document-classification?select=bbc_data.csv$0) was used. The dataset contains article text and category labels. The categories include Sport, Business, Entertainment, Politics, and Tech.

# NLP formulation of the problem

This problem is formulated as a supervised NLP multi-class text classification task. More specifically it is a document classification task since each full article is classified as a whole.

# High-level system design

In this end-to-end text classification system, the input are the raw text of a news article that had been proccessed and transformed into machine-readable features. Several classification models were applied on the features and the output is one predicted category label from a fixed set of classes.

The system uses two approaches: traditional ML pipeline using text preprocessing, TF-IDF features as well as classifiers; and Transformer-based deep learning pipeline using DistilRoBERTa.

# Main components of the system

* Data Ingestion: It loads the dataset as an structured input data so that the models can be trained on them.
* Label encoding: It is required by many ML and DL models to have numerical target labels
* Train-test split: It allows the model to train on one portion of the data and evaluate performance on the other portion. It seperates train set from test set early on to prevent data leakage.
* Text Cleaning: It standardize and remove noise from text so that the models can focus on informative terms and improve feature quality.
* Text preprocessing: It transforms clean text into normalized units. This helps models detect patterns more effectively.
* Feature enginnering for traditional ML: uses TF-IDF to convert text into numerical vectors so that the models can "understand" them!
* Tokenization for Transformer model: It prepares raw text for Transformers architecture by converting text into the format expected by DistilRoBERTa.
* Dataset formatting: It converts pandas dataframeinto Hugging Face dataset.Dataset since their trainer expects structured dataset with fields such as input ID, attention masks and labels.
* Model training: It is the core prediction engine that trains several models and provides baseline for comparison.
* Predition: It uses the trained models to assign a label to new unseen data. It is the operational goal of the whole system.
* Evaluation: It measures how well the models perform to determine whether the system is useful and to identify which models perform the best.

# How the components are connected

Traditional ML pipeline:

→ Data loading → Label encoding → Train-test split → Text cleaning → Text preprocessing → TF-IDF vectorization → Train ML classifiers → Predict category → Evaluate


Transformer pipeline:

→ Data loading → Label encoding → Train-test split → Tokenization with pretrained tokenizer → Dataset formatting → Fine-tune DistilRoBERTa sequence classifier → Predict category for new text → Evaluate


In [1]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import sklearn.preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
import datasets
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import transformers

import evaluate
from sklearn.metrics import classification_report

In [3]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

# Data Ingestion

Loads the dataset and give an overview of it

In [4]:
df = pd.read_csv("/content/bbc_data.csv")

In [5]:
df

,data,labels
0,Musicians to tackle US red tape Musicians gro...,entertainment
1,"U2s desire to be number one U2, who have won ...",entertainment
2,Rocker Doherty in on-stage fight Rock singer ...,entertainment
3,Snicket tops US box office chart The film ada...,entertainment
4,"Oceans Twelve raids box office Oceans Twelve,...",entertainment
...,...,...
2220,Warning over Windows Word files Writing a Mic...,tech
2221,Fast lifts rise into record books Two high-sp...,tech
2222,Nintendo adds media playing to DS Nintendo is...,tech
2223,Fast moving phone viruses appear Security fir...,tech


In [6]:
df['labels'].value_counts()

,count
labels,
sport,511
business,510
politics,417
tech,401
entertainment,386


Label Encoder turns the textual label of each class in target column into a number so that the models can be train on them.

In [7]:
le = sklearn.preprocessing.LabelEncoder()
df["labels"] = le.fit_transform(df["labels"])
number_of_classes = df["labels"].nunique()
df.head()

,data,labels
0,Musicians to tackle US red tape Musicians gro...,1
1,"U2s desire to be number one U2, who have won ...",1
2,Rocker Doherty in on-stage fight Rock singer ...,1
3,Snicket tops US box office chart The film ada...,1
4,"Oceans Twelve raids box office Oceans Twelve,...",1


In [8]:
df['labels'].value_counts()

,count
labels,
3,511
0,510
2,417
4,401
1,386


In [9]:
# Splits the dataset into train and test sets and prints their shape
dataset_train, dataset_test = sklearn.model_selection.train_test_split(df, test_size=0.2)

print("Dataset shape: ", df.shape)
print("Train set shape: ", dataset_train.shape)
print("Test set shape: ", dataset_test.shape)

Dataset shape:  (2225, 2)
Train set shape:  (1780, 2)
Test set shape:  (445, 2)


Seperates target column from the rest of the dataset in both train and test sets so that the models can learn the patterns on the training data and then be examined on the test set without access to the "answers" (target coulmn).

In [10]:
x_train = dataset_train.drop("labels", axis=1)
y_train = dataset_train['labels']

x_test = dataset_test.drop("labels", axis=1)
y_test = dataset_test['labels']

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(1780, 1)
(1780,)
(445, 1)
(445,)


# Text Cleaning

Defines a function to clean the text. This include lower casing all letters, removing URLs, usernames such as @john, characters that are not letters, and finally returns a text with removed extra spaces from the beginning and end of the string. The cleaned text is stored in another column in the set called "clean_text" to preserve the original text untouched.

In [11]:
def dataset_cleaner(text):
  text = text.lower()
  text = re.sub(r"http\S+", "", text)
  text = re.sub(r"http", "", text)
  text = re.sub(r"@\S+", "", text)
  text = re.sub(r"[^a-zA-Z]", " ", text)
  return text.strip()


x_train["clean_text"] = x_train['data'].apply(dataset_cleaner)

x_train.head()

,data,clean_text
2150,Concerns over Windows ATMs Cash machine netwo...,concerns over windows atms cash machine netwo...
1545,Schools to take part in mock poll Record numb...,schools to take part in mock poll record numb...
1405,OGara revels in Ireland victory Ireland fly-h...,ogara revels in ireland victory ireland fly h...
976,Prodigy Monfils blows away Gaudio French prod...,prodigy monfils blows away gaudio french prod...
626,BA to suspend two Saudi services British Airw...,ba to suspend two saudi services british airw...


# Text Preprocessing

Chooses the pretrained model for Transformer's pipeline. Creates a tokenizer to convert text into numbers so that the model can understand. It pads sequences in a batch to the same length because different texts have different lengths. Defines a function to apply the previous preprocessing steps on the data. Then it converts the pandas dataframe into Hugging Face Dataset object. Finally it removes the "data" and "__index_level_0__" column as their presence cause error in later steps.


In [12]:
model_name = "distilroberta-base"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
data_collator = transformers.DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
    return tokenizer(examples["data"], truncation=True, padding=True, max_length=512)

df_train_encoded = datasets.Dataset.from_pandas(dataset_train).map(preprocess_function, batched=True)
df_test_encoded = datasets.Dataset.from_pandas(dataset_test).map(preprocess_function, batched=True)

df_train_encoded = df_train_encoded.remove_columns(["data", "__index_level_0__"])
df_test_encoded = df_test_encoded.remove_columns(["data", "__index_level_0__"])

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/1780 [00:00<?, ? examples/s]

Map:   0%|          | 0/445 [00:00<?, ? examples/s]

Creates a nested function that split sentences into words, remove stopwords (such as "and" or "is"), assigns a grammatical tag to each token, then creates a loop to go through each word and its POS tag and lemmatizes the word using the correct POS. Finally it reduces the words to their stem and join all the stems back to make a signle text string. The preprocessed data is stored in a seperate column called "preprocessed_text".

In [13]:
lemmatizer = WordNetLemmatizer()
stemmer = nltk.stem.PorterStemmer()

def text_preprocessor(text):
  tokens = nltk.word_tokenize(text)
  stop_words = set(stopwords.words('english'))
  tagged_tokens = pos_tag(tokens)

  def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('N'):
        return 'n'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'

  lemmatized_sentence = []
  for word, tag in tagged_tokens:
      if word == 'are' or word in ['is', 'am']:
          lemmatized_sentence.append(word)
      else:
          lemmatized_sentence.append(
              lemmatizer.lemmatize(word, get_wordnet_pos(tag)))

  filtered_words = [word for word in lemmatized_sentence if word not in stop_words]
  stems = [stemmer.stem(token) for token in filtered_words]
  return " ".join(stems)

x_train["preprocessed_text"] = x_train["clean_text"].apply(text_preprocessor)

x_train.head()

,data,clean_text,preprocessed_text
2150,Concerns over Windows ATMs Cash machine netwo...,concerns over windows atms cash machine netwo...,concern window atm cash machin network could s...
1545,Schools to take part in mock poll Record numb...,schools to take part in mock poll record numb...,school take part mock poll record number schoo...
1405,OGara revels in Ireland victory Ireland fly-h...,ogara revels in ireland victory ireland fly h...,ogara revel ireland victori ireland fli half r...
976,Prodigy Monfils blows away Gaudio French prod...,prodigy monfils blows away gaudio french prod...,prodigi monfil blow away gaudio french prodigi...
626,BA to suspend two Saudi services British Airw...,ba to suspend two saudi services british airw...,ba suspend two saudi servic british airway hal...


# Feature Engineering

This section is solely for traditional models. It creates a TF-IDF feature extractor to learns word importance from the train set and to converts train and test text into numeric feature.

In [14]:
vectorizer = TfidfVectorizer()

x_train_tfidf = vectorizer.fit_transform(x_train["preprocessed_text"])
x_test_tfidf = vectorizer.transform(x_test['data'])

In [15]:
print(x_train_tfidf.shape)
print(x_test_tfidf.shape)

(1780, 17196)
(445, 17196)


# Model Training

It defines how the Transformer model will be evaluated. First it loads an accuracy calculator, then defines a function that takes the model’s raw outputs and converts them into predicted class labels. Then it compares them with the true labels and returns the accuracy score.

In [16]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

It sets up a Transformer's model for training, defines training settings using TrainingArguments function and creates the trainer that manages batching, forward pass, loss computation, backpropagation, optimization, and evaluation on its own. Finally it trains the model.

In [17]:
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=number_of_classes)

training_args = transformers.TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    report_to="none"
)

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=df_train_encoded,
    eval_dataset=df_test_encoded,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.121399
1000,0.001835


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1120, training_loss=0.05508726086866643, metrics={'train_runtime': 944.9326, 'train_samples_per_second': 18.837, 'train_steps_per_second': 1.185, 'total_flos': 2358045846528000.0, 'train_loss': 0.05508726086866643, 'epoch': 10.0})

The following are KNN, Logistic Regression, Random Forest, SVM, and Decision Tree classifier models being trained and tested.

In [18]:
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(x_train_tfidf, y_train)

KNeighborsClassifier()

In [19]:
knn_pred = knn_model.predict(x_test_tfidf)

In [20]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(x_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [21]:
lr_pred = lr_model.predict(x_test_tfidf)

In [22]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(x_train_tfidf, y_train)

RandomForestClassifier(random_state=42)

In [23]:
rf_pred = rf_model.predict(x_test_tfidf)

In [24]:
svm_model = SVC(kernel='linear')

svm_model.fit(x_train_tfidf, y_train)

SVC(kernel='linear')

In [25]:
svm_pred = svm_model.predict(x_test_tfidf)

In [26]:
dt_model = DecisionTreeClassifier()

dt_model.fit(x_train_tfidf, y_train)

DecisionTreeClassifier()

In [27]:
dt_pred = dt_model.predict(x_test_tfidf)

# Evaluation and Model Comparison

Table below represent class numbers with its respective lable for evaluation metrix of each traditional model.

Class Number | Class Label
-------------|------------
0 | business
1 | entertainment
2 | politics
3 | sport
4 | tech

The accuracy score for each model is represented in the following table:

Model Name | Accuracy Score
-------------|------------
KNN | 0.86
Logistic Regression | 0.90
Random Forest | 0.85
SVM | 0.91
Decision Tree | 0.64
DistilRoBERTa | 0.98



With respect to the above table, SVM and Logistic Regression show a great performance in classifying articles correctly and stand at the top among other traditional ML models, however, (perhaps as expected) Transformer's model, DistilRoBERTa, outperforms the traditional ML models by an accuracy score of 0.98!

A closer look on classification report on traditional model suggest that perhaps some classes may be harder to separate than others, however the challenging class to classify is inconsistent accross models.

Traditional ML models are fast, simple, and cost-effective, however they require more preprocessing and text cleaning. On the other hand, DistilRoBERTa can recognize deeper semantic patterns in the document text, and doesn't require much text cleaning and preprocessing, although it is significantly slower and computationally taxing for the system that runs it.

In [28]:
print("Classification Report:\n")
print(classification_report(y_test, knn_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.74      0.88      0.80        98
           1       0.91      0.79      0.85        73
           2       0.87      0.78      0.82        77
           3       0.91      0.95      0.93       113
           4       0.92      0.86      0.89        84

    accuracy                           0.86       445
   macro avg       0.87      0.85      0.86       445
weighted avg       0.87      0.86      0.86       445



In [29]:
print("Classification Report:\n")
print(classification_report(y_test, lr_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.98      0.90        98
           1       0.95      1.00      0.97        73
           2       0.98      0.71      0.83        77
           3       0.85      0.99      0.92       113
           4       1.00      0.77      0.87        84

    accuracy                           0.90       445
   macro avg       0.92      0.89      0.90       445
weighted avg       0.91      0.90      0.90       445



In [30]:
print("Classification Report:\n")
print(classification_report(y_test, rf_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.79      0.94      0.86        98
           1       0.78      0.99      0.87        73
           2       0.94      0.79      0.86        77
           3       0.88      0.99      0.93       113
           4       0.98      0.51      0.67        84

    accuracy                           0.85       445
   macro avg       0.87      0.84      0.84       445
weighted avg       0.87      0.85      0.84       445



In [31]:
print("Classification Report:\n")
print(classification_report(y_test, svm_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.96      0.90        98
           1       0.88      1.00      0.94        73
           2       0.97      0.81      0.88        77
           3       0.91      0.99      0.95       113
           4       1.00      0.77      0.87        84

    accuracy                           0.91       445
   macro avg       0.92      0.91      0.91       445
weighted avg       0.92      0.91      0.91       445



In [32]:
print("Classification Report:\n")
print(classification_report(y_test, dt_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.62      0.65      0.64        98
           1       0.60      0.78      0.68        73
           2       0.56      0.57      0.57        77
           3       0.72      0.88      0.80       113
           4       0.68      0.25      0.37        84

    accuracy                           0.64       445
   macro avg       0.64      0.63      0.61       445
weighted avg       0.64      0.64      0.62       445



In [33]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
0.001835,0.064222,1120,0.986517


{'eval_loss': 0.06422171741724014, 'eval_accuracy': 0.9865168539325843}

# Conclusion

**Strength**: end-to-end solution, practical, consist of two complementary approaches: Traditional ML pipeline can be used when speed, interpretability and lower infrastructure cost is of higher priority for the company. While the Transformer pipeline can be adopted when maximum predictive quality is the priority.

**Limitation**: The pipeline was only tested on one dataset with significantly different classes in the target. A dataset with semantically closer classes (such as football, handball, basketball, etc; or sports with a ball, sports without a ball, solo playing sports, and team playing sports) might examine the robustness of the pipelines more rigorously.

**Implication**: The results suggest that automated article classification is quite feasible and can significantly reduce manual work and human error, improve consistency, and support recommendation and site organization.

**Data-driven recommendation**: Deploy simpler models for quicker runtime, lower cost. Consider Transformer's models if there is a significant outperform on difficult categories. In general, it is recommended to retrain the systems periodically paired with human monitoring.